# Linear Algebra & Calculus for ML — Hands‑On Demo
### DATA2201: Fall 2025

This notebook provides small, runnable examples that mirror the lecture topics: matrices, vector spaces, eigenvalues/eigenvectors, partial derivatives, and gradient descent. 

**How to use this notebook**
- Run each cell top-to-bottom.
- Read the short explanations in the markdown cells.
- Modify parameters (e.g., matrices, learning rates) and re-run to see effects.


In [ ]:
# Core imports (no seaborn as requested)
import math
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)

## 1) Matrices & Operations

We review basic operations (addition, multiplication, transpose, determinant, inverse) and relate them to ML tasks such as feature transforms.


In [ ]:
# Define two small matrices and perform basic ops
A = np.array([[1, 3],
              [2, 4]], dtype=float)

B = np.array([[2, 1],
              [1, 3]], dtype=float)

C = np.array([[2, 3],
              [1, 4]], dtype=float)

print("A =\n", A)
print("B =\n", B)

print("\nAddition: A + B =\n", A + B)
print("\nMatrix product: A @ B =\n", A @ B)

print("\nTranspose: A.T =\n", A.T)

detC = np.linalg.det(C)
print("\nDeterminant det(C) =", round(detC, 3))

if abs(detC) > 1e-12:
    print("\nInverse of C =\n", np.linalg.inv(C))
else:
    print("\nC is singular; inverse does not exist.")

## 2) Vector Spaces & Linear Independence

A simple way to **test linear independence** numerically is to form a matrix with the vectors as columns and check its rank.
- If rank equals the number of vectors, they are linearly independent.


In [ ]:
# 2D example: two vectors
v1 = np.array([1, 0.0])
v2 = np.array([0, 1.0])

M = np.column_stack([v1, v2])
rank = np.linalg.matrix_rank(M)

print("Vectors as columns:\n", M)
print("Rank =", rank, " — independent?" , rank == M.shape[1])

## 3) Eigenvalues & Eigenvectors (not covered in class)

- We compute eigenvalues/eigenvectors and show how a linear transform stretches/rotates the plane.
- Try changing the matrix `A`.


In [ ]:
# Choose a 2x2 matrix for geometric intuition
A = np.array([[4.0, 1.0],
              [2.0, 3.0]])

vals, vecs = np.linalg.eig(A)
print("Eigenvalues:\n", vals)
print("\nEigenvectors (columns):\n", vecs)


## 4) Partial Derivatives & Gradient — Example

We define a simple quadratic function:

$
f(x, y) = x^2 + y^2
$

Its gradient is:

$
\nabla f(x, y) = [2x, 2y]
$

The gradient points in the direction of **steepest increase** of $ f $.
For this function, the gradient always points directly **away from the origin**,
and its magnitude increases with distance from the origin.

Below, we compute $ f(x, y) $ and $ \nabla f(x, y) $ at a few sample points.


In [ ]:

# Define function and gradient
def f(x, y):
    return x**2 + y**2

def grad_f(x, y):
    return np.array([2*x, 2*y])

# Sample points
samples = np.array([
    [-2.0, -1.5],
    [1.5, -1.0],
    [2.0, 2.0],
    [-1.0, 2.0]
])

print("f(x, y) = x^2 + y^2")
print("Gradient ∇f(x, y) = [2x, 2y]\n")

# Compute and display
for (x, y) in samples:
    val = f(x, y)
    g = grad_f(x, y)
    print(f"At (x, y) = ({x:5.2f}, {y:5.2f}) → f = {val:5.2f}, ∇f = {g}")

print("\nEach gradient points directly away from the origin (direction of steepest increase).")


### Chain Rule (Numeric Check)

Let $z = x^2 + y^2$ with $x=t^2,\ y=3t$. Then 
$$\frac{dz}{dt} = 2x\frac{dx}{dt} + 2y\frac{dy}{dt} = 4t^3 + 18t.$$

We verify numerically via finite differences.


In [ ]:
def z_of_t(t):
    x = t**2
    y = 3*t
    return x**2 + y**2

def dz_dt_chain(t):
    return 4*(t**3) + 18*t

def finite_diff(f, t, h=1e-6):
    return (f(t+h) - f(t-h)) / (2*h)

ts = np.linspace(-2, 2, 9)
table = []
for t in ts:
    exact = dz_dt_chain(t)
    approx = finite_diff(z_of_t, t)
    table.append((t, exact, approx, abs(exact-approx)))

print(" t     exact_dz/dt   finite_diff    abs_error")
for row in table:
    print(f"{row[0]: .2f}   {row[1]: .6f}    {row[2]: .6f}   {row[3]: .2e}")

## 5) Gradient Descent (1D)

We minimize $J(\theta) = (\theta - 3)^2$ with learning rate \(\eta=0.1\).

In [ ]:
def J(theta):
    return (theta - 3.0)**2

def dJ(theta):
    return 2.0 * (theta - 3.0)

def gradient_descent(theta0=0.0, lr=0.1, n_iter=15):
    thetas = [theta0]
    for _ in range(n_iter):
        thetas.append(thetas[-1] - lr * dJ(thetas[-1]))
    thetas = np.array(thetas)
    return thetas, J(thetas)

thetas, costs = gradient_descent()
iterations = np.arange(len(thetas))
gradients = dJ(thetas)

# Print values for iterations 0–10
print("Iteration  theta      grad        cost")
for k in iterations:
    print(f"{k:3d}      {thetas[k]:7.3f}   "
          f"{gradients[k]:7.3f}   {costs[k]:7.3f}")

# Plot theta, gradient, and cost
fig, ax = plt.subplots(figsize=(8, 6))

ax.plot(iterations, thetas, "bo-", label=r"$\theta$",
        linewidth=2, markersize=8)
ax.plot(iterations, gradients, "rs--", label=r"$\nabla J(\theta)$",
        linewidth=2, markersize=8)
ax.plot(iterations, costs, "g^-.", label=r"$J(\theta)$",
        linewidth=2, markersize=8)

ax.set_xlabel("Iteration", fontsize=16)
ax.set_ylabel("Values", fontsize=16)
ax.set_title("Iteration Values for θ, Gradient, and Cost", fontsize=18)
ax.set_xticks(np.arange(0, 11, 2))
ax.set_yticks(np.arange(-6, 10, 2))
ax.tick_params(labelsize=13)
ax.grid(True, linestyle="--", alpha=0.7)
ax.legend(fontsize=16, loc="upper right")

plt.tight_layout()
plt.show()

# See You Next Lecture!